# Colab runner — badminton stroke study
Run **Section 0** in every session. Sections 1–2 run once (Day 2). Results, configs and checkpoints live in the repo on Google Drive; frames live on local disk for speed.

Edit the two paths marked `EDIT`.

## Section 0 — setup (every session)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/badminton'          # EDIT: project folder on Drive
REPO = f'{DRIVE}/repo'                              # this repository, uploaded to Drive
VIDEO_ZIP = f'{DRIVE}/VideoBadminton.zip'           # EDIT: downloaded dataset archive
%cd {REPO}
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, torch, torchvision
print(torch.__version__, torchvision.__version__, 'CPUs:', os.cpu_count())
FR = 'data.frames_root=/content/frames'             # used by every training command

In [ ]:
# restore frames in a new session (after Section 2 has been run once)
!test -d /content/frames || tar -xf {DRIVE}/frames.tar -C /content
!ls /content/frames | wc -l

## Section 1 — index and split (Day 2, once)
1. Unzip and inspect file names. 2. Choose the session key (**decision**). 3. Write index and splits.

In [ ]:
!mkdir -p /content/vb && unzip -q -n {VIDEO_ZIP} -d /content/vb
!python -m src.make_index --video-root /content/vb --inspect

In [ ]:
SESSION_KEY = 'date_time'   # DECIDE from --inspect: 'date' or 'date_time'
!python -m src.make_index --video-root /content/vb --session-key {SESSION_KEY}
!python -m src.make_splits

## Section 2 — frame extraction (Day 2, once)

In [ ]:
!python -m src.extract_frames --video-root /content/vb --frames-root /content/frames --workers {os.cpu_count()}
!tar -cf {DRIVE}/frames.tar -C /content frames
!grep -c -E 'unreadable|decode_failed' data/frame_meta.csv || true

## Section 3 — Day 3: smoke test, calibration, profiling

In [ ]:
!python -m src.train --config configs/base.yaml --smoke --override {FR}

In [ ]:
!python -m src.calibrate --override {FR}

In [ ]:
!python -m src.profile_model --device cuda

## Section 3b — LR check (approved group L, 3 runs)
After it finishes, write the selected LR into `train.lr` in `configs/base.yaml` **before** generating group A.

In [ ]:
!python -m src.make_configs --group L
!python -m src.run_group --group L --override {FR}
!python -m src.analyze lrcheck

## Section 4 — run plan (Days 4–7)
Each cell is resumable: re-run it after a disconnect.

In [ ]:
!python -m src.make_configs --group A
!python -m src.run_group --group A --override {FR}

In [ ]:
!python -m src.make_configs --group B
!python -m src.run_group --group B --override {FR}

In [ ]:
!python -m src.make_configs --group C
!cat configs/C/selection.json
!python -m src.run_group --group C --override {FR}

In [ ]:
THIRD = 'B_{bb}_sport_aware'   # DECIDE after C: third key configuration for extra seeds
!python -m src.make_configs --group D --third '{THIRD}'
!python -m src.run_group --group D --override {FR}

In [ ]:
!python -m src.make_configs --group E
!python -m src.run_group --group E --override {FR}

## Section 5 — analysis

In [ ]:
!python -m src.analyze summary
!python -m src.analyze seeds
!python -m src.analyze leakage
for bb in ['mobilenet_v3_large', 'efficientnet_b0', 'resnet18']:
    !python -m src.analyze bootstrap --a A_{bb}_full_s0 --b C_{bb}_opt_s0